# Пара 5. Ключевые слова: текст диссертации vs корпус сравнения

**Вопрос:** какие слова в вашем материале значимо чаще, чем в фоне (Бунин / Чехов / второй автор диссертации)?


## Старт в Colab
1. **Файл → Сохранить копию на Диске** (чтобы правки не пропали).
2. Справа сверху — **Подключить**.
3. Ячейки сверху вниз: Play или Shift+Enter.
Базовый Colab **бесплатный**. Правило занятия: **вопрос → процедура → картинка → цитата → ограничение**.


In [ ]:
import re, math, requests
from collections import Counter
import pandas as pd

BASE_B = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/bunin-temnye-allei/txt/"
BASE_C = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/chekhov-rasskazy/txt/"

def tokenize(t):
    return re.findall(r"[А-Яа-яЁё]+", t.lower())

def load(url):
    r = requests.get(url, timeout=60); r.encoding = "utf-8"; return r.text

# target: замените на свой файл Path("glava1.txt").read_text(encoding="utf-8")
target = load(BASE_C + "dama-s-sobachkoj.txt")
# reference: другой рассказ / Бунин / свой контрольный корпус
reference = load(BASE_B + "chistyj-ponedelnik.txt")

STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто человек когда".split())

def counts(text):
    return Counter(w for w in tokenize(text) if w not in STOP and len(w) > 2)

ct, cr = counts(target), counts(reference)
vocab = set(ct) | set(cr)
N_t, N_r = sum(ct.values()) or 1, sum(cr.values()) or 1

rows = []
for w in vocab:
    ft, fr = ct[w], cr[w]
    # простой contrast: доля в target / (доля в ref + eps)
    score = (ft / N_t) / (fr / N_r + 1e-9)
    if ft >= 3:
        rows.append((w, ft, fr, round(score, 2)))
rows = sorted(rows, key=lambda x: -x[3])[:25]
pd.DataFrame(rows, columns=["слово", "в_target", "в_ref", "contrast"])


## Сдать
Топ-10 кандидатов → вручную вычеркните мусор → **3 слова** оставьте в диссертационный черновик с цитатой.  
Ограничение: без статистики значимости это эвристика, не p-value.
